In [1]:
from pathlib import Path
from typing import Dict, List, Set, Tuple
from docplex.mp.dvar import Var
from docplex.mp.model import Model

Arco = Tuple[str, str]


def ler_instancia_pcm(
    caminho: Path,
) -> Tuple[List[str], Dict[Arco, float], str, str]:
    linhas = [
        linha.strip()
        for linha in caminho.read_text(encoding="utf-8").splitlines()
        if linha.strip()
    ]
    partes = linhas[0].split()
    n_arestas = int(partes[1])
    origem = partes[2]
    destino = partes[3]
    nao_direcionado = partes[4].upper() == "N"

    nos_set: Set[str] = {origem, destino}
    arcos: Dict[Arco, float] = {}

    for linha in linhas[1 : 1 + n_arestas]:
        u, v, custo_str = linha.split()
        custo = float(custo_str)
        nos_set.update([u, v])
        arcos[(u, v)] = custo
        if nao_direcionado:
            arcos[(v, u)] = custo

    return sorted(nos_set), arcos, origem, destino


nos, arcos, origem, destino = ler_instancia_pcm(Path("in.txt"))
print(
    f"Instância carregada: {len(nos)} nós, {len(arcos)} arcos direcionados "
    f"(origem: {origem}, destino: {destino})."
)

Instância carregada: 7 nós, 24 arcos direcionados (origem: A, destino: G).


In [2]:
mdl = Model(name="Problema_do_Caminho_Minimo")

# Variáveis binárias x_ij = 1 se o arco (i, j) faz parte do caminho mínimo
x: Dict[Arco, Var] = {
    (u, v): mdl.binary_var(name=f"x_{u}{v}") for (u, v) in arcos
}

# (1) Função Objetivo: minimizar o custo total do caminho
mdl.minimize(mdl.sum(custo * x[(u, v)] for (u, v), custo in arcos.items()))

# (2) Conservação de fluxo (formulação do slide 8 da Aula 19):
#     saída - entrada = 1 na origem, -1 no destino, 0 nos demais nós
for no in nos:
    fluxo_saida = mdl.sum(x[(u, v)] for (u, v) in arcos if u == no)
    fluxo_entrada = mdl.sum(x[(u, v)] for (u, v) in arcos if v == no)

    if no == origem:
        balanco = 1
    elif no == destino:
        balanco = -1
    else:
        balanco = 0

    mdl.add_constraint(fluxo_saida - fluxo_entrada == balanco, ctname=f"fluxo_{no}")

print("Modelo PCM construído com sucesso.")

Modelo PCM construído com sucesso.


In [3]:
sol = mdl.solve(log_output=False)

if sol is None:
    print("Nenhuma solução encontrada.")
else:
    print(f"Status: {mdl.solve_details.status}")
    print(f"Função Objetivo (Custo Mínimo): {sol.objective_value:.2f}\n")

    arcos_ativos: List[Arco] = []
    for (u, v) in sorted(arcos):
        if int(round(sol.get_value(x[(u, v)]))) == 1:
            arcos_ativos.append((u, v))

    # Reconstrução do caminho da origem ao destino
    proximo: Dict[str, str] = {u: v for u, v in arcos_ativos}
    caminho: List[str] = [origem]
    atual = origem
    while atual != destino and atual in proximo:
        atual = proximo[atual]
        caminho.append(atual)

    print("Caminho mínimo encontrado:", " -> ".join(caminho))
    print("\nArcos escolhidos (x_ij = 1):")
    for u, v in arcos_ativos:
        print(f"  x[{u}->{v}] = 1   (custo: {arcos[(u, v)]:.0f})")

Status: integer optimal solution
Função Objetivo (Custo Mínimo): 22.00

Caminho mínimo encontrado: A -> B -> D -> C -> G

Arcos escolhidos (x_ij = 1):
  x[A->B] = 1   (custo: 10)
  x[B->D] = 1   (custo: 5)
  x[C->G] = 1   (custo: 4)
  x[D->C] = 1   (custo: 3)
